# Creating the Combined Tourism and Climate Dataset (2000–2024)

This notebook reproduces the country-year dataset used for the ISYE 6414 regression project. It downloads and combines international tourism arrivals, annual country-level climate indicators, and reported climate-related disasters.

**Unit of observation:** one country in one year.

**Analysis period:** 2000–2024. Tourism data from 1999 are downloaded only to calculate tourism growth for 2000.

**Sources**

- Tourism arrivals: UN Tourism, processed by Our World in Data
- Climate indicators: World Bank Climate Change Knowledge Portal, ERA5 0.25-degree country annual time series
- Disaster counts: EM-DAT Country Profiles, downloaded through the Humanitarian Data Exchange (HDX)


## 1. Import packages and set project options

If a package is missing, install it with `pip install pandas numpy openpyxl`.

In [ ]:
import io
import json
from functools import reduce
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

START_YEAR = 2000
END_YEAR = 2024
OUTPUT_PATH = Path("project/tourism_climate_country_year_2000_2024.csv")

TOURISM_URL = (
    "https://ourworldindata.org/grapher/international-tourist-trips.csv"
    "?v=1&csvType=full&useColumnShortNames=false"
)
HDX_PACKAGE_URL = (
    "https://data.humdata.org/api/3/action/package_show"
    "?id=emdat-country-profiles"
)
CCKP_API_BASE = "https://cckpapi.worldbank.org/api/v1"

# cws from the initial project notes is interpreted as CWD: consecutive wet days.
CLIMATE_VARIABLES = {
    "tas": "annual_mean_temperature_c",
    "pr": "annual_precipitation_mm",
    "hurs": "annual_mean_relative_humidity_percent",
    "fd": "frost_days_count",
    "tr": "tropical_nights_count",
    "rx1day": "maximum_1day_precipitation_mm",
    "tnn": "annual_minimum_temperature_c",
    "txx": "annual_maximum_temperature_c",
    "rx5day": "maximum_5day_precipitation_mm",
    "cdd": "consecutive_dry_days_count",
    "cwd": "consecutive_wet_days_count",
    "hd30": "hot_days_above_30c_count",
}

USER_AGENT = "ISYE-6414-course-project/1.0"

## 2. Helper functions

The CCKP API returns one nested JSON object per climate variable. The helper below converts that structure into a tidy country-year table.

In [ ]:
def download_bytes(url, timeout=180):
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=timeout) as response:
        return response.read()


def get_json(url, timeout=180):
    return json.loads(download_bytes(url, timeout=timeout).decode("utf-8"))


def fetch_climate_variable(variable_code, output_column):
    query = "_".join(
        [
            "era5-x0.25",
            "timeseries",
            variable_code,
            "timeseries",
            "annual",
            "1950-2025",
            "mean",
            "historical",
            "era5",
            "x0.25",
            "mean",
        ]
    )
    url = f"{CCKP_API_BASE}/{query}/global_countries?_format=json"
    payload = get_json(url)

    if payload.get("metadata", {}).get("status") != "success":
        raise RuntimeError(f"CCKP did not return usable data for {variable_code}")

    records = []
    for country_code, observations in payload["data"].items():
        for period, value in (observations or {}).items():
            year = int(str(period)[:4])
            if START_YEAR <= year <= END_YEAR:
                records.append(
                    {
                        "country_code": country_code,
                        "year": year,
                        output_column: pd.to_numeric(value, errors="coerce"),
                    }
                )

    result = pd.DataFrame(records)
    if result.duplicated(["country_code", "year"]).any():
        raise ValueError(f"Duplicate country-year climate records for {variable_code}")
    return result

## 3. Download and prepare tourism arrivals

Tourism growth is the response variable. It is calculated as a log difference:

$$100[\log(Arrivals_{c,t}+1)-\log(Arrivals_{c,t-1}+1)]$$

The self-join below requires the immediately preceding year to be present. It does not incorrectly use an older observation when a year is missing.

In [ ]:
tourism_raw = pd.read_csv(io.BytesIO(download_bytes(TOURISM_URL)))
tourism_raw.head()

In [ ]:
tourism = tourism_raw.rename(
    columns={
        "Entity": "country",
        "Code": "country_code",
        "Year": "year",
        "Arrivals of tourists from abroad": "tourism_arrivals",
    }
)[["country_code", "country", "year", "tourism_arrivals"]].copy()

tourism["year"] = pd.to_numeric(tourism["year"], errors="coerce")
tourism["tourism_arrivals"] = pd.to_numeric(
    tourism["tourism_arrivals"], errors="coerce"
)
tourism = tourism.dropna(subset=["country_code", "year", "tourism_arrivals"])
tourism = tourism[tourism["country_code"].str.fullmatch(r"[A-Z]{3}")].copy()
tourism["year"] = tourism["year"].astype(int)

previous = tourism[["country_code", "year", "tourism_arrivals"]].copy()
previous["year"] = previous["year"] + 1
previous = previous.rename(
    columns={"tourism_arrivals": "previous_year_tourism_arrivals"}
)

tourism_growth = tourism.merge(
    previous, on=["country_code", "year"], how="inner", validate="one_to_one"
)
tourism_growth = tourism_growth[
    tourism_growth["year"].between(START_YEAR, END_YEAR)
].copy()
tourism_growth["tourism_growth_log_percent"] = 100 * (
    np.log(tourism_growth["tourism_arrivals"] + 1)
    - np.log(tourism_growth["previous_year_tourism_arrivals"] + 1)
)

tourism_growth.shape, tourism_growth.head()

## 4. Download and combine the climate predictors

Each climate table has one row per country-year. The tables are merged using the ISO3 country code and year.

In [ ]:
climate_tables = []
for variable_code, output_column in CLIMATE_VARIABLES.items():
    print(f"Downloading {variable_code}: {output_column}")
    climate_tables.append(fetch_climate_variable(variable_code, output_column))

climate = reduce(
    lambda left, right: left.merge(
        right, on=["country_code", "year"], how="outer", validate="one_to_one"
    ),
    climate_tables,
)
climate = climate.sort_values(["country_code", "year"]).reset_index(drop=True)

print("Climate rows:", len(climate))
print("Climate countries:", climate["country_code"].nunique())
print("Missing values by climate column:")
display(climate.isna().sum())

## 5. Download and aggregate climate-related disasters

The HDX API is used to locate the most recent official EM-DAT Country Profiles workbook. Storm, flood, and drought event counts are summed for each country-year. A country-year without one of these reported events is assigned a count of zero after the left join.

In [ ]:
hdx_metadata = get_json(HDX_PACKAGE_URL)
xlsx_resources = [
    resource
    for resource in hdx_metadata["result"]["resources"]
    if str(resource.get("format", "")).upper() == "XLSX"
]
if not xlsx_resources:
    raise RuntimeError("The HDX package did not contain an XLSX resource.")

disaster_url = xlsx_resources[0]["url"]
print("EM-DAT resource:", disaster_url)
disaster_raw = pd.read_excel(io.BytesIO(download_bytes(disaster_url)))
disaster_raw.head()

In [ ]:
disasters = disaster_raw.copy()
disasters["Year"] = pd.to_numeric(disasters["Year"], errors="coerce")
disasters["Total Events"] = pd.to_numeric(
    disasters["Total Events"], errors="coerce"
)
disasters = disasters[
    disasters["Disaster Type"].isin(["Storm", "Flood", "Drought"])
    & disasters["Year"].between(START_YEAR, END_YEAR)
    & disasters["ISO"].astype(str).str.fullmatch(r"[A-Z]{3}")
].copy()

disaster_counts = (
    disasters.groupby(["ISO", "Year"], as_index=False)["Total Events"]
    .sum()
    .rename(
        columns={
            "ISO": "country_code",
            "Year": "year",
            "Total Events": "climate_disaster_count_storm_flood_drought",
        }
    )
)
disaster_counts["year"] = disaster_counts["year"].astype(int)
disaster_counts.head()

## 6. Join the datasets and create the categorical period variable

The tourism and climate tables use an inner join because the modeled rows require both the response and every climate predictor. Disaster counts use a left join so zero-event country-years are retained.

In [ ]:
combined = tourism_growth.merge(
    climate, on=["country_code", "year"], how="inner", validate="one_to_one"
)
combined = combined.merge(
    disaster_counts,
    on=["country_code", "year"],
    how="left",
    validate="one_to_one",
)
combined["climate_disaster_count_storm_flood_drought"] = (
    combined["climate_disaster_count_storm_flood_drought"].fillna(0)
)

combined["period_category"] = np.select(
    [combined["year"] <= 2019, combined["year"].between(2020, 2021)],
    ["Pre-COVID", "COVID disruption"],
    default="Tourism recovery",
)

FINAL_COLUMNS = [
    "country_code",
    "country",
    "year",
    "period_category",
    "tourism_arrivals",
    "tourism_growth_log_percent",
    "annual_mean_temperature_c",
    "annual_precipitation_mm",
    "annual_mean_relative_humidity_percent",
    "frost_days_count",
    "tropical_nights_count",
    "climate_disaster_count_storm_flood_drought",
    "maximum_1day_precipitation_mm",
    "annual_minimum_temperature_c",
    "annual_maximum_temperature_c",
    "maximum_5day_precipitation_mm",
    "consecutive_dry_days_count",
    "consecutive_wet_days_count",
    "hot_days_above_30c_count",
]

combined = (
    combined[FINAL_COLUMNS]
    .dropna(subset=FINAL_COLUMNS)
    .sort_values(["country_code", "year"])
    .reset_index(drop=True)
)
combined["tourism_growth_log_percent"] = combined[
    "tourism_growth_log_percent"
].round(6)

combined.head()

## 7. Validate the final modeling dataset

These checks verify the project requirements and protect against accidental duplicate keys, missing values, and incorrect period assignments.

In [ ]:
duplicate_count = combined.duplicated(["country_code", "year"]).sum()
missing_count = int(combined.isna().sum().sum())

assert len(combined) >= 3000, "The final modeling dataset has fewer than 3,000 rows."
assert duplicate_count == 0, "Duplicate country-year records were found."
assert missing_count == 0, "Missing values remain in the final dataset."
assert combined["year"].min() == START_YEAR
assert combined["year"].max() == END_YEAR
assert (combined.loc[combined["year"] <= 2019, "period_category"] == "Pre-COVID").all()
assert (
    combined.loc[combined["year"].between(2020, 2021), "period_category"]
    == "COVID disruption"
).all()
assert (combined.loc[combined["year"] >= 2022, "period_category"] == "Tourism recovery").all()

validation_summary = pd.Series(
    {
        "rows": len(combined),
        "columns": len(combined.columns),
        "countries": combined["country_code"].nunique(),
        "minimum_year": combined["year"].min(),
        "maximum_year": combined["year"].max(),
        "duplicate_country_years": duplicate_count,
        "missing_cells": missing_count,
    }
)
display(validation_summary)
display(combined["period_category"].value_counts())
display(combined.groupby("year").size().rename("rows").tail(10))

## 8. Export the full CSV

The CSV is written inside the course `project` folder. Running the notebook from the course root keeps the path relative and portable within this project directory.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(combined):,} rows and {len(combined.columns)} columns to:")
print(OUTPUT_PATH.resolve())

## Notes for the regression report

- Country and year identify the panel observations. They are not weather measurements.
- `period_category` controls for the global tourism disruption and recovery associated with COVID-19.
- The climate fields are spatial country averages. Fractional values are therefore possible for count-style indicators such as frost days and tropical nights.
- EM-DAT counts reported disasters meeting its inclusion criteria, not every weather event. A zero means no qualifying storm, flood, or drought was reported for that country-year in the downloaded country profiles.
- Recent tourism coverage is thinner than earlier coverage, especially in 2023 and 2024. Report the number of available observations by year.
- Because the sources are updated over time, rerunning this notebook later may produce a slightly different row count. Keep the submitted CSV with the project for reproducibility.


## 9. Credit

This dataset was combined by **Affan Rashdi**.